In [0]:
source_schema = 'customer_id long, first_name string, last_name string, email string, phone_no string, address string, city string, state string, zip int'

target_schema = 'customer_id long, first_name string, last_name string, email string, phone_no string, address string, city string, state string, zip int, customer_skey long, effective_to_date date, effective_end_date date, is_current boolean'

max_date = '9999-12-31'
date_format = 'yyyy-MM-dd'

source_data = 'dbfs:/Workspace/Users/pankaj_databricks_outlook.com#ext#@pankajdatabricksoutlook.onmicrosoft.com/PySpark_Practice/data/raw_data'

target_location = '/Volumes/pankajdbws01/default/data/scd_target'

In [0]:
primary_key = ['customer_id']
slowly_changing_column = ['email', 'phone_no', 'address', 'city', 'state', 'zip']
implementation_column = ['customer_skey', 'effective_to_date', 'effective_end_date', 'is_current']

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import datetime

In [0]:
customer_df = spark.read.format('csv').schema(source_schema).option('header', True).load(source_data)

In [0]:
customer_df.display()

In [0]:
window_def = Window.orderBy('customer_id')

In [0]:
ordered_customer_df = customer_df.withColumn('customer_skey', row_number().over(window_def)).withColumn('effective_to_date',current_date()).withColumn('effective_end_date', lit(max_date)).withColumn('is_current', lit(True))

In [0]:
ordered_customer_df.display()

In [0]:
ordered_customer_df.write.format('csv').mode('overwrite').option('header', True).save(target_location)

In [0]:
target_customer_df = spark.read.format('csv').schema((target_schema)).option('header',True).load(target_location)
target_customer_df.display()

In [0]:
max_sk = target_customer_df.select(max('customer_skey')).collect()[0][0]

In [0]:
print(max_sk)

In [0]:
active_customer_df = target_customer_df.filter(col('is_current')==True)
active_customer_df.display()

In [0]:
iactive_customer_df = target_customer_df.filter(col('is_current')!=True)
iactive_customer_df.display()